# 03. VQ Tokenizer Final Gate

목표는 step-02/step-03에서 확인된 boundary codebook 잠식 문제를 기준으로, `kmeans K=32 + boundary discrete token` 비교군 B와 VQ 계열 후보 4종을 같은 입력 계약에서 비교하는 것이다.

검증 범위:
- 8개 dataset: D1 daily 4개, D1 1m 2개, D2 KR 2개
- seeds: 7, 17, 37
- continuous codebook capacity: kmeans/vqvae_latent_kmeans/bsq/coarse_fine K=32, fsq K=30
- 비교 모델: `kmeans_boundary_aware`, `vqvae_latent_kmeans`, `fsq`, `bsq`, `coarse_fine`
- 주 지표: interior-only reconstruction MSE, seed stability, effective vocabulary, boundary/zero token 점유율, D2 지수별 token 점유율 분해

Leakage 규칙:
- scaler, 분위 경계, autoencoder, latent clustering, KMeans는 train split의 `interior x interior` row에만 fit한다.
- boundary row는 학습에서 제외하고 8개 discrete token으로 직접 배정한다.
- zero-range row는 별도 special token으로 배정한다.
- timebox run이므로 hyperparameter sweep은 수행하지 않는다.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import os
from typing import Any

import numpy as np
os.environ.setdefault("MPLCONFIGDIR", "/tmp/kairos-matplotlib")
from matplotlib import pyplot as plt

from kairos.experiments.artifacts import config_hash, find_project_root, write_csv, write_json
from kairos.experiments.protocol import PHASE_01_ID, split_protocol_for_interval
from kairos.experiments.shape_tokenizer.baselines import (
    FEATURE_INPUTS,
    MERGED_FEATURE_INPUTS,
    boundary_aware_fit_rows,
    component_feature_runs,
    dataset_interval,
    fit_boundary_aware_kmeans,
    is_interior_cell,
    latest_feature_run,
    read_merged_shape_rows,
    read_shape_rows,
    reconstruction_mse,
    shape_vectors,
    split_masks,
    token_stats,
)
from kairos.experiments.shape_tokenizer.vq import (
    BSQConfig,
    CoarseFineConfig,
    FSQConfig,
    LatentClusteringConfig,
    build_step03_model_configs,
    fit_bsq,
    fit_coarse_fine,
    fit_fsq,
    fit_vqvae_latent_clustering,
    token_share_by_symbol,
)


In [2]:
PROJECT_ROOT = find_project_root()
RUNS_ROOT = PROJECT_ROOT / "notebooks" / "runs" / "candlestick-shape-quantization"
STEP_03_ID = "step-03-vq-tokenizer"
SOURCE_NOTEBOOK = "notebooks/candlestick-shape-quantization/03_vq_tokenizer.ipynb"

SEEDS = (7, 17, 37)
MODEL_ORDER = (
    "kmeans_boundary_aware",
    "vqvae_latent_kmeans",
    "fsq",
    "bsq",
    "coarse_fine",
)
K = 32
VQ_CONFIG = LatentClusteringConfig(codebook_size=K, epochs=80, hidden_dim=32, latent_dim=4)
FSQ_CONFIG = FSQConfig(levels=(6, 5), epochs=80, hidden_dim=32)
BSQ_CONFIG = BSQConfig(bits=5, epochs=80, hidden_dim=32)
COARSE_FINE_CONFIG = CoarseFineConfig(body_bins=4, fine_per_coarse=4)
EXPECTED_CONTINUOUS_CAPACITY = {
    "kmeans_boundary_aware": 32,
    "vqvae_latent_kmeans": 32,
    "fsq": 30,
    "bsq": 32,
    "coarse_fine": 32,
}

_feature_by_id = {item.dataset_id: item for item in FEATURE_INPUTS}
D1_TARGETS = (
    _feature_by_id["d1_kospi_daily"],
    _feature_by_id["d1_kosdaq_daily"],
    _feature_by_id["d1_nasdaq_daily"],
    _feature_by_id["d1_spx_daily"],
    _feature_by_id["d1_kospi_1m"],
    _feature_by_id["d1_kosdaq_1m"],
)
D2_TARGETS = MERGED_FEATURE_INPUTS
DATASET_IDS = tuple(item.dataset_id for item in D1_TARGETS) + tuple(item.dataset_id for item in D2_TARGETS)

DATASET_IDS


('d1_kospi_daily',
 'd1_kosdaq_daily',
 'd1_nasdaq_daily',
 'd1_spx_daily',
 'd1_kospi_1m',
 'd1_kosdaq_1m',
 'd2_kr-kospi-kosdaq_daily',
 'd2_kr-kospi-kosdaq_1m')

In [3]:
def load_dataset_rows(dataset: Any) -> list[dict[str, Any]]:
    if hasattr(dataset, "components"):
        run_dirs = component_feature_runs(RUNS_ROOT, dataset)
        return read_merged_shape_rows(run_dirs)
    run_dir = latest_feature_run(RUNS_ROOT, dataset)
    return read_shape_rows(run_dir)


def interior_row_indices(rows: list[dict[str, Any]]) -> list[int]:
    return [
        index
        for index, row in enumerate(rows)
        if not row["is_zero_range"] and is_interior_cell(row, eps=VQ_CONFIG.eps)
    ]


def boundary_token_count(tokens: np.ndarray, boundary_token_values: set[int]) -> int:
    return sum(1 for token in tokens.tolist() if int(token) in boundary_token_values)


def compact_result(result: dict[str, Any], rows: list[dict[str, Any]]) -> dict[str, Any]:
    vocabulary = result["vocabulary"]
    return {
        key: value
        for key, value in result.items()
        if key not in {"tokens", "interior_tokens", "vocabulary"}
    } | {
        "token_count": len(result["tokens"]),
        "interior_token_count": len(result["interior_tokens"]),
        "vocabulary": {
            "continuous_codebook_size": vocabulary.continuous_codebook_size,
            "boundary_token_offset": vocabulary.boundary_token_offset,
            "zero_range_token": vocabulary.zero_range_token,
            "size": vocabulary.size,
            "boundary_tokens": {f"{cell[0]}:{cell[1]}": token for cell, token in vocabulary.boundary_tokens.items()},
        },
        "token_share_by_symbol": token_share_by_symbol(rows, result["tokens"], vocabulary_size=vocabulary.size),
    }


def evaluate_boundary_aware_kmeans(rows: list[dict[str, Any]], *, seed: int, split) -> dict[str, Any]:
    tokens, recon, vocabulary = fit_boundary_aware_kmeans(
        rows,
        codebook_size=K,
        seed=seed,
        split=split,
        eps=VQ_CONFIG.eps,
    )
    interior_rows = boundary_aware_fit_rows(rows, eps=VQ_CONFIG.eps)
    interior_indices = interior_row_indices(rows)
    interior_vectors = shape_vectors(interior_rows)
    interior_recon = recon[interior_indices]
    interior_tokens = tokens[interior_indices]
    masks = split_masks(interior_rows, split=split)
    split_metrics = {}
    for split_name, mask in masks.items():
        split_metrics[split_name] = {
            "interior_row_count": int(mask.sum()),
            "interior_reconstruction_mse": reconstruction_mse(interior_vectors[mask], interior_recon[mask]),
            **token_stats(interior_tokens[mask], codebook_size=vocabulary.continuous_codebook_size),
        }
    boundary_count = boundary_token_count(tokens, set(vocabulary.boundary_tokens.values()))
    zero_range_count = int((tokens == vocabulary.zero_range_token).sum())
    return {
        "model": "kmeans_boundary_aware",
        "seed": seed,
        "codebook_size": K,
        "vocabulary_size": vocabulary.size,
        "config": {
            "codebook_size": K,
            "eps": VQ_CONFIG.eps,
            "fit_rows": "train split and interior x interior only",
        },
        "train_interior_row_count": int(masks["train"].sum()),
        "interior_row_count": len(interior_rows),
        "row_count": len(rows),
        "final_train_loss": None,
        "interior_reconstruction_mse": reconstruction_mse(interior_vectors, interior_recon),
        "boundary_token_count": boundary_count,
        "boundary_token_ratio": boundary_count / len(rows) if rows else None,
        "zero_range_token_count": zero_range_count,
        "zero_range_token_ratio": zero_range_count / len(rows) if rows else None,
        "token_usage": token_stats(tokens, codebook_size=vocabulary.size),
        "split_metrics": split_metrics,
        "tokens": tokens,
        "interior_tokens": interior_tokens,
        "vocabulary": vocabulary,
    }


In [4]:
EXPERIMENT_CONFIG = {
    "source_notebook": SOURCE_NOTEBOOK,
    "phase": PHASE_01_ID,
    "step": STEP_03_ID,
    "datasets": list(DATASET_IDS),
    "seeds": list(SEEDS),
    "model_order": list(MODEL_ORDER),
    "comparison": {
        "baseline_b": "kmeans K=32 fit on train interior rows; boundary 8 discrete tokens; zero-range special token",
        "candidate_vqvae_latent_kmeans": "autoencoder trained on train interior rows; KMeans K=32 on encoder latent",
        "candidate_fsq": "autoencoder latent_dim=2 with tanh-bound FSQ levels [6, 5]; continuous capacity 30",
        "candidate_bsq": "autoencoder latent_dim=5 with binary spherical quantization; continuous capacity 32",
        "candidate_coarse_fine": "rule-based direction/body-quartile coarse 8 x KMeans fine 4; continuous capacity 32",
        "timebox": "single fixed-spec run; no hyperparameter sweep",
    },
    "metrics": [
        "interior_reconstruction_mse",
        "seed_std_of_interior_reconstruction_mse",
        "boundary_token_ratio",
        "zero_range_token_ratio",
        "token_usage",
        "token_share_by_symbol_for_d2",
    ],
    "model_configs_by_dataset": {
        dataset_id: build_step03_model_configs(
            dataset_id,
            vq_config=VQ_CONFIG,
            fsq_config=FSQ_CONFIG,
            bsq_config=BSQ_CONFIG,
            coarse_fine_config=COARSE_FINE_CONFIG,
        )
        for dataset_id in DATASET_IDS
    },
}
CFG_HASH = config_hash(EXPERIMENT_CONFIG)
STARTED_AT = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
RUN_DIR = RUNS_ROOT / PHASE_01_ID / STEP_03_ID / "all-datasets-multi" / f"cfg-{CFG_HASH}" / f"run-{STARTED_AT}_seed-multi"
RUN_DIR


PosixPath('/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-03-vq-tokenizer/all-datasets-multi/cfg-d59bafed/run-20260707-032508_seed-multi')

In [5]:
def evaluate_dataset(dataset: Any) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    dataset_id = dataset.dataset_id
    rows = load_dataset_rows(dataset)
    split = split_protocol_for_interval(dataset_interval(dataset_id))
    metrics_rows: list[dict[str, Any]] = []
    summary_rows: list[dict[str, Any]] = []

    for seed in SEEDS:
        raw_results = [
            evaluate_boundary_aware_kmeans(rows, seed=seed, split=split),
            fit_vqvae_latent_clustering(rows, seed=seed, split=split, config=VQ_CONFIG),
            fit_fsq(rows, seed=seed, split=split, config=FSQ_CONFIG),
            fit_bsq(rows, seed=seed, split=split, config=BSQ_CONFIG),
            fit_coarse_fine(rows, seed=seed, split=split, config=COARSE_FINE_CONFIG),
        ]

        for raw_result in raw_results:
            result = compact_result(raw_result, rows)
            result_row = {"dataset_id": dataset_id, **result}
            metrics_rows.append(result_row)
            summary_rows.append(
                {
                    "dataset_id": dataset_id,
                    "model": result["model"],
                    "seed": seed,
                    "row_count": result["row_count"],
                    "token_count": result["token_count"],
                    "interior_row_count": result["interior_row_count"],
                    "interior_token_count": result["interior_token_count"],
                    "train_interior_row_count": result.get("train_interior_row_count"),
                    "codebook_size": result["codebook_size"],
                    "vocabulary_size": result["vocabulary_size"],
                    "interior_reconstruction_mse": result["interior_reconstruction_mse"],
                    "boundary_token_ratio": result["boundary_token_ratio"],
                    "zero_range_token_ratio": result["zero_range_token_ratio"],
                    "effective_vocab_size": result["token_usage"]["effective_vocab_size"],
                    "dead_token_count": result["token_usage"]["dead_token_count"],
                    "token_entropy": result["token_usage"]["token_entropy"],
                    "train_reconstruction_mse": result["split_metrics"]["train"]["interior_reconstruction_mse"],
                    "validation_reconstruction_mse": result["split_metrics"].get("validation", {}).get("interior_reconstruction_mse"),
                    "test_reconstruction_mse": result["split_metrics"].get("test", {}).get("interior_reconstruction_mse"),
                }
            )
    return metrics_rows, summary_rows


def run_vq_validation() -> dict[str, Any]:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    write_json(RUN_DIR / "experiment_config.json", EXPERIMENT_CONFIG | {"cfg_hash": CFG_HASH})
    all_metrics: list[dict[str, Any]] = []
    all_summary: list[dict[str, Any]] = []
    for dataset in (*D1_TARGETS, *D2_TARGETS):
        metrics_rows, summary_rows = evaluate_dataset(dataset)
        all_metrics.extend(metrics_rows)
        all_summary.extend(summary_rows)
    write_csv(RUN_DIR / "tables" / "summary.csv", all_summary)
    result = {
        "experiment_config": EXPERIMENT_CONFIG | {"cfg_hash": CFG_HASH},
        "run_dir": str(RUN_DIR),
        "metrics": all_metrics,
        "summary": all_summary,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(RUN_DIR / "metrics.json", result)
    return result


In [6]:
RUN_RESULTS = run_vq_validation()
RUN_RESULTS["summary"][:10]


[{'dataset_id': 'd1_kospi_daily',
  'model': 'kmeans_boundary_aware',
  'seed': 7,
  'row_count': 9420,
  'token_count': 9420,
  'interior_row_count': 6838,
  'interior_token_count': 6838,
  'train_interior_row_count': 2379,
  'codebook_size': 32,
  'vocabulary_size': 41,
  'interior_reconstruction_mse': 0.4724463819272994,
  'boundary_token_ratio': 0.2740976645435244,
  'zero_range_token_ratio': 0.0,
  'effective_vocab_size': 30.4571449271785,
  'dead_token_count': 3,
  'token_entropy': 4.928708803867729,
  'train_reconstruction_mse': 0.384839191831843,
  'validation_reconstruction_mse': 0.4011501220717384,
  'test_reconstruction_mse': 0.42891938727290463},
 {'dataset_id': 'd1_kospi_daily',
  'model': 'vqvae_latent_kmeans',
  'seed': 7,
  'row_count': 9420,
  'token_count': 9420,
  'interior_row_count': 6838,
  'interior_token_count': 6838,
  'train_interior_row_count': 2379,
  'codebook_size': 32,
  'vocabulary_size': 41,
  'interior_reconstruction_mse': 0.49484671068151526,
  'bound

In [7]:
def _summary_rows(results: dict[str, Any]) -> list[dict[str, Any]]:
    return list(results["summary"])


def _mean_by_dataset_model(rows: list[dict[str, Any]], metric: str) -> dict[tuple[str, str], tuple[float, float]]:
    grouped: dict[tuple[str, str], list[float]] = {}
    for row in rows:
        value = row.get(metric)
        if value is None:
            continue
        grouped.setdefault((row["dataset_id"], row["model"]), []).append(float(value))
    return {
        key: (float(np.mean(values)), float(np.std(values)))
        for key, values in grouped.items()
    }


def _bar_offsets(models: list[str]) -> tuple[np.ndarray, float]:
    width = min(0.8 / len(models), 0.16)
    offsets = (np.arange(len(models)) - (len(models) - 1) / 2) * width
    return offsets, width


def plot_reconstruction_mse(summary_rows: list[dict[str, Any]], figure_dir: Path) -> Path:
    datasets = list(dict.fromkeys(row["dataset_id"] for row in summary_rows))
    models = list(MODEL_ORDER)
    stats = _mean_by_dataset_model(summary_rows, "interior_reconstruction_mse")
    x = np.arange(len(datasets))
    offsets, width = _bar_offsets(models)
    fig, ax = plt.subplots(figsize=(15, 5.5))
    for offset, model in zip(offsets, models, strict=True):
        means = [stats.get((dataset, model), (np.nan, np.nan))[0] for dataset in datasets]
        stds = [stats.get((dataset, model), (np.nan, np.nan))[1] for dataset in datasets]
        ax.bar(x + offset, means, width, yerr=stds, label=model, capsize=2)
    ax.set_title("Interior reconstruction MSE by dataset")
    ax.set_ylabel("MSE in shape-core space")
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=35, ha="right")
    ax.legend(frameon=False, ncols=3)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    path = figure_dir / "reconstruction_mse_by_dataset.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def plot_effective_vocab(summary_rows: list[dict[str, Any]], figure_dir: Path) -> Path:
    datasets = list(dict.fromkeys(row["dataset_id"] for row in summary_rows))
    models = list(MODEL_ORDER)
    stats = _mean_by_dataset_model(summary_rows, "effective_vocab_size")
    x = np.arange(len(datasets))
    offsets, width = _bar_offsets(models)
    fig, ax = plt.subplots(figsize=(15, 5.5))
    for offset, model in zip(offsets, models, strict=True):
        means = [stats.get((dataset, model), (np.nan, np.nan))[0] for dataset in datasets]
        ax.bar(x + offset, means, width, label=model)
    ax.set_title("Effective vocabulary size by dataset")
    ax.set_ylabel("exp(token entropy)")
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=35, ha="right")
    ax.legend(frameon=False, ncols=3)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    path = figure_dir / "effective_vocab_by_dataset.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def plot_boundary_zero_ratios(summary_rows: list[dict[str, Any]], figure_dir: Path) -> Path:
    datasets = list(dict.fromkeys(row["dataset_id"] for row in summary_rows))
    models = list(MODEL_ORDER)
    boundary_stats = _mean_by_dataset_model(summary_rows, "boundary_token_ratio")
    zero_stats = _mean_by_dataset_model(summary_rows, "zero_range_token_ratio")
    x = np.arange(len(datasets))
    offsets, width = _bar_offsets(models)
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    for ax, stats, title in [
        (axes[0], boundary_stats, "Boundary token ratio"),
        (axes[1], zero_stats, "Zero-range token ratio"),
    ]:
        for offset, model in zip(offsets, models, strict=True):
            means = [stats.get((dataset, model), (np.nan, np.nan))[0] for dataset in datasets]
            ax.bar(x + offset, means, width, label=model)
        ax.set_title(title)
        ax.set_ylabel("row ratio")
        ax.grid(axis="y", alpha=0.25)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(datasets, rotation=35, ha="right")
    axes[0].legend(frameon=False, ncols=3)
    fig.tight_layout()
    path = figure_dir / "boundary_zero_ratio_by_dataset.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def plot_d2_symbol_token_heatmap(results: dict[str, Any], figure_dir: Path) -> Path:
    rows = []
    for row in results["metrics"]:
        if not row["dataset_id"].startswith("d2_") or row["seed"] != SEEDS[0]:
            continue
        for symbol, shares in row["token_share_by_symbol"].items():
            rows.append((row["dataset_id"], row["model"], symbol, shares))
    max_vocab = max(EXPECTED_CONTINUOUS_CAPACITY.values()) + 9
    matrix = np.zeros((len(rows), max_vocab), dtype=float)
    labels = []
    for row_index, (dataset_id, model, symbol, shares) in enumerate(rows):
        labels.append(f"{dataset_id} | {model} | {symbol}")
        for token_id, share in shares.items():
            token_index = int(token_id)
            if token_index < max_vocab:
                matrix[row_index, token_index] = float(share)
    fig_height = max(6, 0.35 * len(rows))
    fig, ax = plt.subplots(figsize=(14, fig_height))
    image = ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap="viridis")
    ax.set_title(f"D2 symbol token share heatmap (seed {SEEDS[0]})")
    ax.set_xlabel("token id")
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)
    fig.colorbar(image, ax=ax, label="share")
    fig.tight_layout()
    path = figure_dir / "d2_symbol_token_share_heatmap_seed7.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def plot_seed_mse_std(summary_rows: list[dict[str, Any]], figure_dir: Path) -> Path:
    datasets = list(dict.fromkeys(row["dataset_id"] for row in summary_rows))
    models = list(MODEL_ORDER)
    stats = _mean_by_dataset_model(summary_rows, "interior_reconstruction_mse")
    x = np.arange(len(datasets))
    offsets, width = _bar_offsets(models)
    fig, ax = plt.subplots(figsize=(15, 5.5))
    for offset, model in zip(offsets, models, strict=True):
        stds = [stats.get((dataset, model), (np.nan, np.nan))[1] for dataset in datasets]
        ax.bar(x + offset, stds, width, label=model)
    ax.set_title("Seed std of interior reconstruction MSE")
    ax.set_ylabel("std across seeds")
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=35, ha="right")
    ax.legend(frameon=False, ncols=3)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    path = figure_dir / "seed_mse_std_by_dataset.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def make_figures(results: dict[str, Any], run_dir: Path) -> list[str]:
    figure_dir = run_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = _summary_rows(results)
    paths = [
        plot_reconstruction_mse(summary_rows, figure_dir),
        plot_effective_vocab(summary_rows, figure_dir),
        plot_boundary_zero_ratios(summary_rows, figure_dir),
        plot_d2_symbol_token_heatmap(results, figure_dir),
        plot_seed_mse_std(summary_rows, figure_dir),
    ]
    return [str(path) for path in paths]


In [8]:
FIGURE_FILES = make_figures(RUN_RESULTS, RUN_DIR)
RUN_RESULTS["figure_files"] = FIGURE_FILES
write_json(RUN_DIR / "metrics.json", RUN_RESULTS)
FIGURE_FILES


['/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-03-vq-tokenizer/all-datasets-multi/cfg-d59bafed/run-20260707-032508_seed-multi/figures/reconstruction_mse_by_dataset.png',
 '/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-03-vq-tokenizer/all-datasets-multi/cfg-d59bafed/run-20260707-032508_seed-multi/figures/effective_vocab_by_dataset.png',
 '/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-03-vq-tokenizer/all-datasets-multi/cfg-d59bafed/run-20260707-032508_seed-multi/figures/boundary_zero_ratio_by_dataset.png',
 '/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-03-vq-tokenizer/all-datasets-multi/cfg-d59bafed/run-20260707-032508_seed-multi/figures/d2_symbol_token_share_heatmap_seed7.png',
 '/Users/choeseoggyu/portfolio/kairos/notebooks/

In [9]:
# D2 지수별 token 점유율 분해 확인
d2_breakdown = [
    {
        "dataset_id": row["dataset_id"],
        "model": row["model"],
        "seed": row["seed"],
        "token_share_by_symbol": row["token_share_by_symbol"],
    }
    for row in RUN_RESULTS["metrics"]
    if row["dataset_id"].startswith("d2_")
]
d2_breakdown[:5]


[{'dataset_id': 'd2_kr-kospi-kosdaq_daily',
  'model': 'kmeans_boundary_aware',
  'seed': 7,
  'token_share_by_symbol': {'KOSPI': {'0': 0.0065817409766454355,
    '1': 0.035668789808917196,
    '2': 0.027176220806794056,
    '3': 0.05615711252653928,
    '4': 0.02303609341825902,
    '5': 0.045859872611464965,
    '6': 0.006475583864118896,
    '7': 0.008280254777070064,
    '8': 0.008386411889596603,
    '9': 0.028768577494692145,
    '10': 0.03641188959660297,
    '11': 0.019532908704883226,
    '12': 0.008174097664543524,
    '13': 0.0321656050955414,
    '14': 0.024946921443736732,
    '15': 0.011146496815286623,
    '16': 0.025902335456475585,
    '17': 0.007112526539278132,
    '18': 0.014861995753715499,
    '19': 0.04670912951167728,
    '20': 0.024840764331210193,
    '21': 0.01910828025477707,
    '22': 0.01740976645435244,
    '23': 0.04777070063694268,
    '24': 0.0020169851380042463,
    '25': 0.015817409766454352,
    '26': 0.03789808917197452,
    '27': 0.049044585987261

In [10]:
# 기본 무결성 체크: 8 datasets x 3 seeds x 5 models
assert len(DATASET_IDS) == 8
assert len(RUN_RESULTS["summary"]) == len(DATASET_IDS) * len(SEEDS) * len(MODEL_ORDER)
assert {row["model"] for row in RUN_RESULTS["summary"]} == set(MODEL_ORDER)
assert all(row["token_count"] == row["row_count"] for row in RUN_RESULTS["summary"])
assert all(row["interior_token_count"] == row["interior_row_count"] for row in RUN_RESULTS["summary"])
assert all(row["vocabulary_size"] == row["codebook_size"] + 9 for row in RUN_RESULTS["summary"])
assert all(row["interior_reconstruction_mse"] is not None for row in RUN_RESULTS["summary"])
for model, expected_capacity in EXPECTED_CONTINUOUS_CAPACITY.items():
    rows = [row for row in RUN_RESULTS["summary"] if row["model"] == model]
    assert rows
    assert all(row["codebook_size"] == expected_capacity for row in rows), model
assert (RUN_DIR / "experiment_config.json").exists()
assert (RUN_DIR / "metrics.json").exists()
assert (RUN_DIR / "tables" / "summary.csv").exists()
assert len(FIGURE_FILES) == 5
assert all(Path(path).exists() for path in FIGURE_FILES)
